# 06 — Métricas de Desempenho

Avaliação no conjunto de teste dos três modelos treinados no notebook `05_cnn_classification.ipynb`:

1. CNN (`models/modelo_cnn.keras`)
2. FCNN (`models/modelo_fcnn.keras`)
3. Encoder-Decoder (`models/modelo_encdec.keras`)

Para cada modelo: métricas de classificação, matriz de confusão, curvas ROC e Precision-Recall e análise dos erros. No final, uma tabela compara os três modelos.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             average_precision_score, roc_curve, precision_recall_curve,
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.preprocessing import label_binarize

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

## Carregamento do conjunto de teste

In [ ]:
splits = pd.read_csv(PROJECT_ROOT / 'data' / 'annotations' / 'splits.csv')
classes = sorted(splits['classe'].unique())
teste = splits[splits['conjunto'] == 'teste'].reset_index(drop=True)

X_test, y_true = [], []
for _, row in teste.iterrows():
    caminho = PROJECT_ROOT / row['caminho'].replace('spectrograms', 'mod_spectograms')
    img = Image.open(caminho).convert('RGB').resize((224, 224))
    X_test.append(np.array(img) / 255.0)
    y_true.append(classes.index(row['classe']))
X_test = np.array(X_test, dtype='float32')
y_true = np.array(y_true)
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])

print(classes, X_test.shape)

## Funções de avaliação

As mesmas funções são usadas para os três modelos.

In [ ]:
def prever(arquivo):
    """Carrega o modelo e retorna as probabilidades da Softmax para o conjunto de teste."""
    model = tf.keras.models.load_model(PROJECT_ROOT / 'models' / arquivo)
    return model.predict(X_test, verbose=0)


def metricas(y_pred):
    """Acurácia, Precisão, Recall e F1-Score por classe."""
    print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))
    relatorio = classification_report(y_true, y_pred, target_names=classes, zero_division=0, output_dict=True)
    return pd.DataFrame(relatorio).T.round(2)


def matriz_confusao(y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel('Classe Prevista')
    plt.ylabel('Classe Real')
    plt.title(f'Matriz de Confusão — {titulo}')
    plt.show()


def curvas_roc_pr(y_pred_prob, titulo):
    """ROC-AUC e PR-AUC globais e por classe, com as curvas."""
    print('ROC-AUC global:', round(roc_auc_score(y_true_bin, y_pred_prob, multi_class='ovr'), 4))
    print('PR-AUC global: ', round(average_precision_score(y_true_bin, y_pred_prob), 4))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, classe in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
        axes[0].plot(fpr, tpr, label=classe)
        precisao, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_prob[:, i])
        axes[1].plot(recall, precisao, label=classe)
    axes[0].plot([0, 1], [0, 1], 'k--')
    axes[0].set_title(f'Curva ROC — {titulo}')
    axes[0].set_xlabel('Taxa de Falsos Positivos')
    axes[0].set_ylabel('Taxa de Verdadeiros Positivos')
    axes[0].legend()
    axes[1].set_title(f'Curva Precision-Recall — {titulo}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precisão')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    return pd.DataFrame({
        'ROC-AUC': roc_auc_score(y_true_bin, y_pred_prob, average=None),
        'PR-AUC': average_precision_score(y_true_bin, y_pred_prob, average=None),
    }, index=classes).round(2)


def analise_erros(y_pred_prob):
    """Imagens erradas, falsos positivos/negativos por classe e casos ambíguos."""
    y_pred = np.argmax(y_pred_prob, axis=1)

    resultado = pd.DataFrame(y_pred_prob.round(2), columns=classes)
    resultado.insert(0, 'arquivo', teste['caminho'].str.split('/').str[-1])
    resultado.insert(1, 'real', [classes[i] for i in y_true])
    resultado.insert(2, 'prevista', [classes[i] for i in y_pred])
    prob_ordenada = np.sort(y_pred_prob, axis=1)
    resultado['diferenca_top2'] = (prob_ordenada[:, -1] - prob_ordenada[:, -2]).round(2)

    erros = resultado[resultado['real'] != resultado['prevista']]
    print(f'Imagens classificadas incorretamente: {len(erros)} de {len(resultado)}')
    display(erros)

    # Falso Positivo da classe X: previsto como X, mas era outra classe.
    # Falso Negativo da classe X: era X, mas foi previsto como outra classe.
    print('Falsos Positivos e Falsos Negativos por classe:')
    display(pd.DataFrame({
        'Falsos Positivos': [((y_pred == i) & (y_true != i)).sum() for i in range(4)],
        'Falsos Negativos': [((y_true == i) & (y_pred != i)).sum() for i in range(4)],
    }, index=classes))

    # Casos ambíguos: as duas maiores probabilidades diferem menos de 0.15.
    print('Casos ambíguos (diferença entre as duas maiores probabilidades < 0.15):')
    display(resultado[resultado['diferenca_top2'] < 0.15])

    n = min(6, len(erros))
    if n > 0:
        fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
        for ax, idx in zip(np.atleast_1d(axes), erros.index[:n]):
            ax.imshow(X_test[idx])
            ax.set_title(f"Real: {resultado.loc[idx, 'real']}\nPrevista: {resultado.loc[idx, 'prevista']}", fontsize=9)
            ax.axis('off')
        plt.tight_layout()
        plt.show()

# 1. CNN

In [ ]:
prob_cnn = prever('modelo_cnn.keras')
pred_cnn = np.argmax(prob_cnn, axis=1)

## Acurácia, Precisão, Recall e F1-Score

In [ ]:
metricas(pred_cnn)

## Matriz de confusão

In [ ]:
matriz_confusao(pred_cnn, 'CNN')

## Curvas ROC e Precision-Recall

In [ ]:
curvas_roc_pr(prob_cnn, 'CNN')

## Análise sistemática dos erros

In [ ]:
analise_erros(prob_cnn)

### Respostas — CNN

**Qual classe apresentou o maior desempenho?**
_(preencher)_

**Qual classe apresentou maior dificuldade de classificação?**
_(preencher)_

**Quais classes apresentaram maior número de acertos?**
_(preencher)_

**Quais classes foram mais confundidas?**
_(preencher)_

**Existe uma classe particularmente difícil para o modelo?**
_(preencher)_

**O modelo apresenta boa capacidade de separação entre as classes?**
_(preencher)_

**As curvas indicam desempenho consistente?**
_(preencher)_

**Análise dos erros (falsos positivos, falsos negativos e casos ambíguos):**
_(preencher)_

# 2. FCNN

In [ ]:
prob_fcnn = prever('modelo_fcnn.keras')
pred_fcnn = np.argmax(prob_fcnn, axis=1)

## Acurácia, Precisão, Recall e F1-Score

In [ ]:
metricas(pred_fcnn)

## Matriz de confusão

In [ ]:
matriz_confusao(pred_fcnn, 'FCNN')

## Curvas ROC e Precision-Recall

In [ ]:
curvas_roc_pr(prob_fcnn, 'FCNN')

## Análise sistemática dos erros

In [ ]:
analise_erros(prob_fcnn)

### Respostas — FCNN

**Qual classe apresentou o maior desempenho?**
_(preencher)_

**Qual classe apresentou maior dificuldade de classificação?**
_(preencher)_

**Quais classes apresentaram maior número de acertos?**
_(preencher)_

**Quais classes foram mais confundidas?**
_(preencher)_

**Existe uma classe particularmente difícil para o modelo?**
_(preencher)_

**O modelo apresenta boa capacidade de separação entre as classes?**
_(preencher)_

**As curvas indicam desempenho consistente?**
_(preencher)_

**Análise dos erros (falsos positivos, falsos negativos e casos ambíguos):**
_(preencher)_

# 3. Encoder-Decoder

In [ ]:
prob_encdec = prever('modelo_encdec.keras')
pred_encdec = np.argmax(prob_encdec, axis=1)

## Acurácia, Precisão, Recall e F1-Score

In [ ]:
metricas(pred_encdec)

## Matriz de confusão

In [ ]:
matriz_confusao(pred_encdec, 'Encoder-Decoder')

## Curvas ROC e Precision-Recall

In [ ]:
curvas_roc_pr(prob_encdec, 'Encoder-Decoder')

## Análise sistemática dos erros

In [ ]:
analise_erros(prob_encdec)

### Respostas — Encoder-Decoder

**Qual classe apresentou o maior desempenho?**
_(preencher)_

**Qual classe apresentou maior dificuldade de classificação?**
_(preencher)_

**Quais classes apresentaram maior número de acertos?**
_(preencher)_

**Quais classes foram mais confundidas?**
_(preencher)_

**Existe uma classe particularmente difícil para o modelo?**
_(preencher)_

**O modelo apresenta boa capacidade de separação entre as classes?**
_(preencher)_

**As curvas indicam desempenho consistente?**
_(preencher)_

**Análise dos erros (falsos positivos, falsos negativos e casos ambíguos):**
_(preencher)_

# Comparação entre os modelos

Métricas globais no conjunto de teste. Precisão, Recall e F1-Score são a média entre as 4 classes (*macro*). Quanto mais escura a célula, melhor o valor na coluna.

In [ ]:
probabilidades = {'CNN': prob_cnn, 'FCNN': prob_fcnn, 'Encoder-Decoder': prob_encdec}

linhas = []
for nome, prob in probabilidades.items():
    pred = np.argmax(prob, axis=1)
    linhas.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_true, pred),
        'Precisão': precision_score(y_true, pred, average='macro', zero_division=0),
        'Recall': recall_score(y_true, pred, average='macro', zero_division=0),
        'F1-Score': f1_score(y_true, pred, average='macro', zero_division=0),
        'ROC-AUC': roc_auc_score(y_true_bin, prob, multi_class='ovr'),
        'PR-AUC': average_precision_score(y_true_bin, prob),
    })

comparacao = pd.DataFrame(linhas).set_index('Modelo').round(2)
comparacao.style.background_gradient(cmap='Blues', axis=0).format('{:.2f}')

In [ ]:
comparacao.plot(kind='bar', figsize=(12, 5), rot=0)
plt.title('Comparação das métricas no conjunto de teste')
plt.ylabel('Valor')
plt.ylim(0, 1)
plt.axhline(0.25, color='gray', linestyle='--', label='Acaso (4 classes)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

### Conclusão

**Qual modelo teve o melhor desempenho geral?**
_(preencher)_

**Algum modelo ficou claramente acima do acaso (25% de acurácia)?**
_(preencher)_